# Memory Estimator

Оценка VRAM для pretrain/SFT/GRPO: `estimate_memory_footprint`, `get_architecture_profile`.

In [ ]:
from homellm.models.memory_estimator import (
    get_architecture_profile,
    estimate_memory_footprint,
    estimate_grpo_memory_footprint,
    ARCHITECTURE_PROFILES,
)

print('Profiles:', list(ARCHITECTURE_PROFILES.keys()))

In [ ]:
# Конфиг как в студии (hidden_size, num_layers, seq_len, batch_size)
config = {
    'hidden_size': 512,
    'num_layers': 8,
    'n_heads': 8,
    'seq_len': 2048,
    'batch_size': 4,
    'vocab_size': 50257,
    'intermediate_size': 2048,
}

profile = get_architecture_profile('home')
params = profile.calculate_parameters(
    vocab_size=config['vocab_size'],
    hidden_size=config['hidden_size'],
    num_layers=config['num_layers'],
    intermediate_size=config['intermediate_size'],
    max_position_embeddings=config['seq_len'],
)
print(f'Params: {params:,}')

In [ ]:
config['max_position_embeddings'] = config['seq_len']
est = estimate_memory_footprint(config, batch_size=config['batch_size'], num_gpus=1)
for k, v in est.items():
    if isinstance(v, (int, float)) and v > 0:
        print(f'{k}: {v/1024**3:.2f} GB' if v > 1e6 else f'{k}: {v}')

In [ ]:
# GRPO оценка
grpo_config = {
    'hidden_size': 512,
    'num_layers': 8,
    'n_heads': 8,
    'vocab_size': 50257,
    'grpo_group_size': 8,
    'grpo_train_batch_size': 2,
    'max_prompt_length': 512,
    'grpo_max_new_tokens': 1024,
}
grpo_est = estimate_grpo_memory_footprint(grpo_config, num_gpus=1)
print('GRPO estimate keys:', list(grpo_est.keys())[:10])